# Indexing NTCIR-2 AdHoc by OpenSearch for Sparse Encoding Model

Prerequisite
  - [ntcir2-adhoc_preprocess.ipynb](../../dataset/ntcir2-adhoc/ntcir1-adhoc-preprocess.ipynb)
  - [ml_model_registration.ipynb](ml_model_registration.ipynb)


In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas opensearch-py dotenv

In [ ]:
import pprint
from tqdm import tqdm

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

Install opensearch plugins for Japanese processing
  - Requires a restart of docker container after the installation

In [ ]:
!docker exec -it opensearch-node \
  /usr/share/opensearch/bin/opensearch-plugin install --batch analysis-kuromoji analysis-icu
!docker restart opensearch-node

In [ ]:
import os
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'dataset', 'ntcir2-adhoc'))

In [ ]:
import ir_datasets
import ntcir2_adhoc
dataset = ir_datasets.load('ntcir2-adhoc')

In [ ]:
index_name = "ntcir2_splade"

In [ ]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

### Sparse Encoding of Chunks (server-side ingest pipeline)

Chunk **and** sparse-encode entirely inside OpenSearch. Bulk sends *raw* documents; a single ingest pipeline runs two processors in order:

1. `text_chunking` — splits `text` into passages in `text_chunks`.
2. `sparse_encoding` — calls the remote model on each passage and writes per-chunk `rank_features` to the nested `text_chunks_embedding` field.

In [ ]:
pipeline_id = "ntcir2_chunk_sparse"
model_id = "your-model-id"

In [ ]:
def create_chunk_sparse_pipeline(
    pipeline_id: str,
    model_id: str,
    source_field: str = "text",
    chunk_field: str = "text_chunks",
    embedding_field: str = "text_chunks_embedding",
    token_limit: int = 384,
    overlap_rate: float = 0.2,
    tokenizer: str = "standard",
    batch_size: int = 16,
) -> dict:
    """
    Create (or update) an ingest pipeline that chunks then sparse-encodes text,
    fully server-side.

    Stage 1 (`text_chunking`) splits `source_field` into passages in `chunk_field`.
    Stage 2 (`sparse_encoding`) embeds each passage with the remote `model_id`,
    writing a nested list of rank_features to `embedding_field`.

    `batch_size` bundles that many *documents'* chunks into a single model call
    (batch ingestion), so the GPU processes them as one padded batch instead of
    one at a time. Actual texts per call ~= batch_size x avg chunks/doc.
    """
    body = {
        "description": "Chunk documents, then sparse-encode each passage",
        "processors": [
            {
                "text_chunking": {
                    "algorithm": {
                        "fixed_token_length": {
                            "token_limit": token_limit,
                            "overlap_rate": overlap_rate,
                            "tokenizer": tokenizer,
                        }
                    },
                    "field_map": {source_field: chunk_field},
                }
            },
            {
                "sparse_encoding": {
                    "model_id": model_id,
                    "field_map": {chunk_field: embedding_field},
                    "batch_size": batch_size,
                }
            },
        ],
    }
    return client.ingest.put_pipeline(id=pipeline_id, body=body)

response = create_chunk_sparse_pipeline(pipeline_id, model_id, batch_size=64)
pprint.pprint(response)

Bulk indexing

In [ ]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1
    },
    "analysis":{
      "tokenizer": {
        "ja_search": {
            "type": "kuromoji_tokenizer",
            "mode": "search"
        }
      },
      "analyzer": {
        "default": {
            "type": "custom",
            "tokenizer": "ja_search",
            "char_filter": ["icu_normalizer", "kuromoji_iteration_mark"],
            "filter": [
                "kuromoji_baseform",
                "kuromoji_part_of_speech",
                "ja_stop",
                "kuromoji_number",
                "kuromoji_stemmer"
            ]
        }
      }
    }
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
        "text_chunks": { "type": "text" },
        # Per-chunk sparse vectors produced by the sparse_encoding processor.
        # Chunking yields a list of passages, so the embeddings must be `nested`.
        "text_chunks_embedding": {
            "type": "nested",
            "properties": {
                "sparse_encoding": { "type": "rank_features" }
            }
        },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

In [ ]:
def prepare_documents(dataset):
    """
    Yield raw bulk actions. Chunking + sparse-encoding happen server-side in the
    ingest pipeline, so we only ship docid/text/title. Progress is tracked by the
    bulk loop below (via streaming_bulk), not here.
    """
    docstore = dataset.docs_store()

    for doc in dataset.docs_iter():
        # Parse the document
        title, text = docstore.get(doc.doc_id).text.split(None, 1)
        text = text.replace("\n", " ")
        yield {
            "_id": doc.doc_id,  # Unique identifier for the document
            "_source": {
                "docid": doc.doc_id,
                "text": text,
                "title": title
            }
        }

In [ ]:
from opensearchpy.helpers import streaming_bulk

# Batch inference is configured server-side in the pipeline's `sparse_encoding`
# processor (batch_size=64). OpenSearch 3.x removed the `_bulk?batch_size=` query
# param (2.x only), so we must NOT pass one here — it 400s the whole request.

# Total for the progress bar (docs_count is instant; fall back to a full id scan).
try:
    total = dataset.docs_count()
except Exception:
    total = sum(1 for _ in dataset.docs_iter())

success, errors = 0, []
with tqdm(total=total, desc="Indexing") as bar:
    for ok, item in streaming_bulk(
        client,
        prepare_documents(dataset),
        index=index_name,
        pipeline=pipeline_id,
        chunk_size=128,                # 2 x processor batch_size; bar advances every 128 docs
        request_timeout=300,
        max_retries=3,
        initial_backoff=2,
        raise_on_error=False,          # collect failures instead of aborting the run
        raise_on_exception=False,
    ):
        bar.update(1)                  # advances per actually-processed doc
        success += ok
        if not ok:
            errors.append(item)

print(f"indexed: {success},  failed: {len(errors)}")
if errors:
    pprint.pprint(errors[:3])          # inspect the first few errors

---
(Optional) Running indexing again for missing documents

In [ ]:
from opensearchpy.helpers import scan

# All ids actually in the index (~736k, _source disabled -> fast, ~a minute)
indexed = set()
for hit in scan(
    client,
    index=index_name,
    query={"query": {"match_all": {}}, "_source": False},
    size=5000,
):
    indexed.add(hit["_id"])

# All ids the dataset should have produced
all_ids = {doc.doc_id for doc in dataset.docs_iter()}

missing = sorted(all_ids - indexed)
print(f"indexed: {len(indexed)},  missing: {len(missing)}")
print(missing[:10])

In [ ]:
# Re-index the docs identified as missing by the scan diff, printing every error.
def prepare_missing(dataset, missing_ids):
    docstore = dataset.docs_store()
    for doc_id in missing_ids:
        title, text = docstore.get(doc_id).text.split(None, 1)
        text = text.replace("\n", " ")
        yield {
            "_id": doc_id,
            "_source": {"docid": doc_id, "text": text, "title": title},
        }

retry_ok, retry_failed = 0, []
for ok, item in streaming_bulk(
    client,
    prepare_missing(dataset, missing),
    index=index_name,
    pipeline=pipeline_id,
    chunk_size=128,
    request_timeout=300,
    raise_on_error=False,
    raise_on_exception=False,
):
    retry_ok += ok
    if not ok:
        retry_failed.append(item)

print(f"retried ok: {retry_ok}, still failing: {len(retry_failed)}\n")

for item in retry_failed:          # full error detail for every failure
    pprint.pprint(item)
    print("-" * 80)